# Carbon Emission Dataset Analysis

**Advanced Python Project — Second Semester 2025/2026**

This notebook provides a complete solution for the assignment based on the public Kaggle dataset `carbon_emission_dataset_with_Industry.csv`. The work includes data loading, inspection, cleaning, exploratory data analysis, statistical analysis, correlation heatmap, pairplot, boxplots, time-series exploration, moving averages, rolling variance, decomposition, and answers to the required analytical questions.

> The target variable is **`Carbon_Emission_tCO2e_TARGET`**, which measures carbon emissions in tonnes of CO2 equivalent.


In [ ]:
# If you are running this notebook in a fresh environment, uncomment the next line.
# !pip install pandas numpy matplotlib seaborn scipy statsmodels

from pathlib import Path
import io
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')

PROJECT_ROOT = Path('..').resolve() if Path('../data').exists() else Path('.').resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'carbon_emission_dataset_with_Industry.csv'
FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'Carbon_Emission_tCO2e_TARGET'
DATE_COL = 'Date'
REQUIRED_STATS_COLS = [
    'Total_Energy_Consumption_kWh',
    'Renewable_Energy_Consumption_kWh',
    'NonRenewable_Energy_Consumption_kWh',
    'Production_Output_Units',
    'Supply_Chain_Transport_km'
]


## 1. Load and Inspect the Dataset

The first step imports the CSV file, displays the first rows, checks dataset dimensions, prints column information, and calculates descriptive statistics.


In [ ]:
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

display(df.describe(include='all').T)


## 2. Data Cleaning

The dataset is cleaned by converting `Date` to datetime, removing duplicates, checking missing values, filling any unexpected missing values, and creating additional variables for deeper analysis.


In [ ]:
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip()
clean_df[DATE_COL] = pd.to_datetime(clean_df[DATE_COL], errors='coerce')

print('Missing values before cleaning:')
display(clean_df.isna().sum().to_frame('missing_values'))

before_duplicates = clean_df.duplicated().sum()
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print('Duplicate rows removed:', before_duplicates)

numeric_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = clean_df.select_dtypes(exclude=[np.number, 'datetime64[ns]']).columns.tolist()

for col in numeric_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in categorical_cols:
    mode_value = clean_df[col].mode(dropna=True)
    clean_df[col] = clean_df[col].fillna(mode_value.iloc[0] if len(mode_value) else 'Unknown')

clean_df = clean_df.dropna(subset=[DATE_COL]).reset_index(drop=True)

clean_df['Renewable_Share_Actual_Percent'] = clean_df['Renewable_Energy_Consumption_kWh'] / clean_df['Total_Energy_Consumption_kWh'] * 100
clean_df['NonRenewable_Share_Actual_Percent'] = clean_df['NonRenewable_Energy_Consumption_kWh'] / clean_df['Total_Energy_Consumption_kWh'] * 100
clean_df['Emission_Intensity_tCO2e_per_Unit'] = clean_df[TARGET] / clean_df['Production_Output_Units']
clean_df['YearMonth'] = clean_df[DATE_COL].dt.to_period('M').astype(str)
clean_df['Month'] = clean_df[DATE_COL].dt.month
clean_df['DayOfWeek'] = clean_df[DATE_COL].dt.day_name()

clean_df.to_csv(PROCESSED_DIR / 'carbon_emission_cleaned.csv', index=False)

print('Cleaned shape:', clean_df.shape)
print('Missing values after cleaning:', int(clean_df.isna().sum().sum()))
display(clean_df.head())


## 3. Exploratory Data Analysis and Required Basic Statistics

The assignment asks for mean, median, maximum, and minimum for five operational variables. Standard deviation is also included because it supports the variability question.


In [ ]:
basic_stats = clean_df[REQUIRED_STATS_COLS].agg(['mean', 'median', 'min', 'max', 'std']).T
basic_stats = basic_stats.round(4)
display(basic_stats)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 14))
axes = axes.flatten()
for ax, col in zip(axes, REQUIRED_STATS_COLS):
    sns.histplot(clean_df[col], kde=True, bins=30, ax=ax)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
axes[-1].axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_required_feature_distributions.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
categorical_cols_for_bars = ['Sector', 'Industry_Sectors', 'Supply_Chain_Transport_Mode', 'Carbon_Reduction_Strategy']
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, col in zip(axes.flatten(), categorical_cols_for_bars):
    order = clean_df[col].value_counts().index
    sns.countplot(data=clean_df, x=col, order=order, ax=ax)
    ax.set_title(f'Frequency of {col}')
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_categorical_frequency_bars.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
daily_energy = clean_df.groupby(DATE_COL, as_index=False)[[
    'Total_Energy_Consumption_kWh',
    'Renewable_Energy_Consumption_kWh',
    'NonRenewable_Energy_Consumption_kWh'
]].mean()

plt.figure(figsize=(15, 7))
for col in daily_energy.columns.drop(DATE_COL):
    sns.lineplot(data=daily_energy, x=DATE_COL, y=col, label=col)
plt.title('Average Energy Consumption Over Time')
plt.xlabel('Date')
plt.ylabel('Average kWh')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_energy_line_over_time.png', dpi=180, bbox_inches='tight')
plt.show()


## 4. Scatter Plots and Combined Relationships

The PDF requests scatter plots of emissions versus temperature and emissions versus time. The provided CSV does **not** contain a temperature column, so the notebook explicitly checks for a temperature field and only plots it if available. The emissions-versus-time plot is generated from the available data.


In [ ]:
sample_df = clean_df.sample(min(3000, len(clean_df)), random_state=42)

plt.figure(figsize=(15, 6))
sns.scatterplot(data=sample_df, x=DATE_COL, y=TARGET, hue='Industry_Sectors', alpha=0.55)
plt.title('Carbon Emissions vs Time')
plt.xlabel('Date')
plt.ylabel('Carbon Emission (tCO2e)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_emissions_vs_time_scatter.png', dpi=180, bbox_inches='tight')
plt.show()

temperature_columns = [c for c in clean_df.columns if 'temp' in c.lower()]
if temperature_columns:
    temp_col = temperature_columns[0]
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=sample_df, x=temp_col, y=TARGET, alpha=0.5)
    plt.title(f'Carbon Emissions vs {temp_col}')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'notebook_emissions_vs_temperature_scatter.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('No temperature column exists in this CSV; therefore, an emissions-vs-temperature scatter plot cannot be generated from the provided dataset.')


In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=sample_df,
    x='Renewable_Energy_Consumption_kWh',
    y='NonRenewable_Energy_Consumption_kWh',
    hue=TARGET,
    palette='magma',
    alpha=0.65
)
plt.title('Renewable vs Non-Renewable Energy Consumption Colored by Emissions')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_emission_source_pair_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


## 5. Bivariate and Multivariate Analysis

This section includes the required correlation heatmap, boxplot of emissions grouped by industry sector, pairplot of selected features, and group-based analysis by sector/category.


In [ ]:
numeric_df = clean_df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(17, 13))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.4)
plt.title('Correlation Heatmap of Numerical Variables')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_correlation_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

display(corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False).head(10).to_frame('Correlation with Target'))


In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=clean_df, x='Industry_Sectors', y=TARGET)
plt.title('Carbon Emissions by Industry Sector')
plt.xlabel('Industry Sector')
plt.ylabel('Carbon Emission (tCO2e)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_emissions_boxplot_by_industry.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
selected_pairplot_cols = [
    TARGET,
    'Total_Energy_Consumption_kWh',
    'Renewable_Energy_Consumption_kWh',
    'NonRenewable_Energy_Consumption_kWh',
    'Production_Output_Units',
    'Supply_Chain_Transport_km'
]

pair_sample = clean_df[selected_pairplot_cols + ['Industry_Sectors']].sample(min(1500, len(clean_df)), random_state=42)
g = sns.pairplot(pair_sample, vars=selected_pairplot_cols, hue='Industry_Sectors', corner=True, plot_kws={'alpha': 0.55, 's': 18})
g.fig.suptitle('Pairplot of Selected Features', y=1.02)
g.savefig(FIG_DIR / 'notebook_selected_features_pairplot.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
sector_summary = clean_df.groupby('Sector')[TARGET].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(4)
industry_summary = clean_df.groupby('Industry_Sectors')[TARGET].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(4)

display(sector_summary)
display(industry_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=clean_df, x='Sector', y=TARGET, estimator='mean', errorbar=None, ax=axes[0])
axes[0].set_title('Average Emissions by Sector')
axes[0].tick_params(axis='x', rotation=30)
sns.barplot(data=clean_df, x='Industry_Sectors', y=TARGET, estimator='mean', errorbar=None, ax=axes[1])
axes[1].set_title('Average Emissions by Industry Sector')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_group_based_emissions.png', dpi=180, bbox_inches='tight')
plt.show()


## 6. Time-Series Exploration, Moving Averages, Rolling Variance, and Decomposition

The `Date` column is already converted to datetime. The analysis aggregates emissions by date, calculates moving averages, calculates rolling variance, and decomposes the daily average emissions series into observed, trend, seasonal, and residual components.


In [ ]:
daily_ts = clean_df.groupby(DATE_COL, as_index=False)[TARGET].mean().sort_values(DATE_COL)
daily_ts['Emissions_7day_MA'] = daily_ts[TARGET].rolling(window=7, min_periods=1).mean()
daily_ts['Emissions_30day_MA'] = daily_ts[TARGET].rolling(window=30, min_periods=1).mean()
daily_ts['Emissions_30day_Rolling_Variance'] = daily_ts[TARGET].rolling(window=30, min_periods=2).var()
daily_ts.to_csv(PROCESSED_DIR / 'daily_time_series.csv', index=False)

display(daily_ts.head())


In [ ]:
plt.figure(figsize=(15, 7))
sns.lineplot(data=daily_ts, x=DATE_COL, y=TARGET, label='Daily average emissions', alpha=0.7)
sns.lineplot(data=daily_ts, x=DATE_COL, y='Emissions_7day_MA', label='7-day moving average')
sns.lineplot(data=daily_ts, x=DATE_COL, y='Emissions_30day_MA', label='30-day moving average')
plt.title('Carbon Emissions Over Time with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Average Carbon Emission (tCO2e)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_emissions_time_series_moving_averages.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(15, 6))
sns.lineplot(data=daily_ts, x=DATE_COL, y='Emissions_30day_Rolling_Variance', color='darkred')
plt.title('30-Day Rolling Variance of Carbon Emissions')
plt.xlabel('Date')
plt.ylabel('Rolling Variance')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_rolling_variance.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
ts = daily_ts.set_index(DATE_COL)[TARGET].asfreq('D').interpolate(method='time')
decomposition = seasonal_decompose(ts, model='additive', period=30)
fig = decomposition.plot()
fig.set_size_inches(15, 10)
fig.suptitle('Additive Time-Series Decomposition of Daily Average Emissions', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_time_series_decomposition.png', dpi=180, bbox_inches='tight')
plt.show()

seasonal_strength = 1 - (np.nanvar(decomposition.resid) / np.nanvar(decomposition.resid + decomposition.seasonal))
print('Estimated seasonal strength:', round(seasonal_strength, 4))


## 7. Required Analytical Questions

This section directly answers the five assignment questions using computed evidence from the dataset.


In [ ]:
# Q1: Which emission sources have the highest variability?
variability_cols = [
    'Total_Energy_Consumption_kWh',
    'Renewable_Energy_Consumption_kWh',
    'NonRenewable_Energy_Consumption_kWh',
    'Supply_Chain_Transport_km',
    'Raw_Material_Usage_kg'
]
variability = pd.DataFrame({
    'mean': clean_df[variability_cols].mean(),
    'std': clean_df[variability_cols].std()
})
variability['coefficient_of_variation'] = variability['std'] / variability['mean']
variability = variability.sort_values('coefficient_of_variation', ascending=False)
display(variability.round(4))
print('Q1 Answer: The highest variability by coefficient of variation is:', variability.index[0])


In [ ]:
# Q2: Are emissions normally distributed?
target_sample = clean_df[TARGET].sample(min(5000, len(clean_df)), random_state=42)
normal_test = stats.normaltest(target_sample)
skewness = clean_df[TARGET].skew()
kurtosis = clean_df[TARGET].kurtosis()

plt.figure(figsize=(10, 6))
sns.histplot(clean_df[TARGET], kde=True, bins=35)
plt.title('Distribution of Carbon Emissions')
plt.xlabel('Carbon Emission (tCO2e)')
plt.tight_layout()
plt.show()

print('Skewness:', round(skewness, 4))
print('Kurtosis:', round(kurtosis, 4))
print('Normality test statistic:', round(normal_test.statistic, 4))
print('Normality test p-value:', normal_test.pvalue)
print('Q2 Answer:', 'Reject normality at alpha=0.05.' if normal_test.pvalue < 0.05 else 'Do not reject normality at alpha=0.05.')


In [ ]:
# Q3: Which variables are strongly correlated?
strong_pairs = []
for i, col1 in enumerate(corr.columns):
    for col2 in corr.columns[i+1:]:
        value = corr.loc[col1, col2]
        if abs(value) >= 0.70:
            strong_pairs.append((col1, col2, value))

strong_pairs_df = pd.DataFrame(strong_pairs, columns=['Variable 1', 'Variable 2', 'Correlation']).sort_values('Correlation', key=lambda s: s.abs(), ascending=False)
if len(strong_pairs_df):
    display(strong_pairs_df.round(4))
else:
    print('No numerical variable pair has an absolute correlation >= 0.70.')

print('Top correlations with target:')
display(corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False).head(10).to_frame('correlation'))


In [ ]:
# Q4: Is there a trend in emissions?
x = np.arange(len(daily_ts))
slope, intercept = np.polyfit(x, daily_ts[TARGET].values, 1)
first_30 = daily_ts[TARGET].iloc[:30].mean()
last_30 = daily_ts[TARGET].iloc[-30:].mean()
pct_change = (last_30 - first_30) / first_30 * 100

print('Linear slope (tCO2e/day):', round(slope, 6))
print('First 30-day average:', round(first_30, 4))
print('Last 30-day average:', round(last_30, 4))
print('Percent change:', round(pct_change, 4), '%')
print('Q4 Answer:', 'There is a modest upward trend.' if slope > 0 else 'There is a modest downward trend.' if slope < 0 else 'The trend is approximately flat.')


In [ ]:
# Q5: Do seasonal patterns exist?
print('Seasonal strength:', round(seasonal_strength, 4))
print('Q5 Answer:', 'A meaningful seasonal component exists.' if seasonal_strength >= 0.30 else 'Only a weak seasonal component is detected.')


## 8. Conclusion

The dataset is clean and suitable for analysis. Carbon emissions are most closely related to energy usage, carbon tax, and related operational variables. The time-series analysis shows a modest annual trend and a detectable seasonal component. The only assignment item that cannot be directly generated is the temperature-related visualization because the provided CSV does not include a temperature column.
